# 05_word_embeddings: Matrix Lookup Equivalence and Skip-Gram with Negative Sampling Loss

This notebook validates word embedding models. It proves the mathematical equivalence between discrete index lookups and linear matrix multiplications, and implements a custom Skip-Gram with Negative Sampling (SGNS) loss module in PyTorch matching our hand-calculations.

## 1. Index Lookup in PyTorch

In [1]:
import torch
import torch.nn as nn
import numpy as np

torch.manual_seed(42)

vocab_size = 5
embed_dim = 3

# Define PyTorch embedding weight matrix W
embedding = nn.Embedding(vocab_size, embed_dim)
W = embedding.weight.data.clone()

idx = 2
idx_tensor = torch.tensor([idx])

# Method 1: Standard index lookup
vector_lookup = embedding(idx_tensor)
print("Embedding Weight Matrix W:\n", W.numpy())
print(f"\nIndex Lookup vector (Index {idx}):", vector_lookup.detach().numpy().flatten())

Embedding Weight Matrix W:
 [[ 0.33669037  0.1288094   0.23446237]
 [ 0.23033303 -1.1228564  -0.18632829]
 [ 2.2082014  -0.63799703  0.46165723]
 [ 0.26735088  0.53490466  0.8093572 ]
 [ 1.1102903  -1.689799   -0.9889599 ]]

Index Lookup vector (Index 2): [ 2.2082014  -0.63799703  0.46165723]


### Output Analysis: Index Lookup
We initialized an embedding layer of size $(5, 3)$ and fetched the vector at index `2`. The weight matrix rows represent the parameters for each word. The output is a single 3-dimensional vector.

## 2. Matrix Multiplication Equivalence

In [2]:
# Method 2: Matrix multiplication (x^T * W) using one-hot vector
x = torch.zeros(vocab_size)
x[idx] = 1.0  # Set index 2 to 1.0

vector_matmul = torch.matmul(x, W)
print(f"Matrix Matmul vector (x^T * W):", vector_matmul.numpy())

# Verify exact equivalence
assert torch.allclose(vector_lookup[0], vector_matmul), "Equivalence check failed!"

Matrix Matmul vector (x^T * W): [ 2.2082014  -0.63799703  0.46165723]


### Output Analysis: Matrix Equivalence
By defining a one-hot vector `x` with `1.0` at index `2` and performing a vector-matrix multiplication $\mathbf{x}^T \mathbf{W}$, we extract the exact same row vector as the lookup. This proves that embedding layers are mathematically identical to standard linear layers with one-hot inputs, allowing gradients to propagate back to update the weights.

## 3. Custom Skip-Gram with Negative Sampling (SGNS) Loss Module

In [3]:
class SGNSLoss(nn.Module):
    def __init__(self):
        super().__init__()
        
    def forward(self, v_target_ctx, v_input, v_neg_ctxs):
        # Positive pair score: log(sigmoid(v_target_ctx . v_input))
        pos_score = torch.sum(v_target_ctx * v_input, dim=-1)
        pos_loss = torch.log(torch.sigmoid(pos_score))
        
        # Negative pairs score: sum(log(sigmoid(-v_neg_ctx . v_input)))
        # v_neg_ctxs shape: (batch_size, k, embed_dim), v_input: (batch_size, embed_dim)
        # We use batch matrix multiplication (bmm) to get dot products for each negative sample
        neg_score = torch.bmm(v_neg_ctxs, v_input.unsqueeze(-1)).squeeze(-1)
        neg_loss = torch.sum(torch.log(torch.sigmoid(-neg_score)), dim=-1)
        
        loss = - (pos_loss + neg_loss)
        return loss.mean()

### Output Analysis: SGNS Module Definition
The custom `SGNSLoss` module computes the Skip-Gram with Negative Sampling loss. It passes the positive pair through a sigmoid function to maximize their alignment, and passes the negative pairs through a negated sigmoid function to push their representations apart in the vector space.

## 4. SGNS Loss Evaluation & Hand-Calculation Consistency

In [4]:
# Initialize inputs matching our hand-calculations exactly
v_input = torch.tensor([[0.1, 0.8, -0.2]], requires_grad=True)
v_target_ctx = torch.tensor([[0.2, 0.9, 0.1]])
v_neg_ctxs = torch.tensor([[[ -0.9, 0.1, 0.8]]])

loss_fn = SGNSLoss()
loss = loss_fn(v_target_ctx, v_input, v_neg_ctxs)
print(f"Initial SGNS Loss: {loss.item():.4f}")

# Verify exact consistency with hand calculation (1.0083)
np.testing.assert_almost_equal(loss.item(), 1.0083, decimal=4)

Initial SGNS Loss: 1.0083


### Output Analysis: Loss Value Verification
The computed initial loss is exactly `1.0083`. This matches our manual calculation where we computed the positive score sigmoid log $-\log \sigma(0.72) \approx 0.3966$ and the negative score sigmoid log $-\log \sigma(0.17) \approx 0.6117$ to get `1.0083`.

## 5. SGNS Gradient Optimization Step

In [5]:
# Execute gradient step
loss.backward()
optimizer = torch.optim.SGD([v_input], lr=0.1)
optimizer.step()

new_loss = loss_fn(v_target_ctx, v_input, v_neg_ctxs)
print(f"Updated SGNS Loss: {new_loss.item():.4f}")

assert new_loss < loss, "Loss did not decrease after gradient optimization step!"

Updated SGNS Loss: 0.9689


### Output Analysis: Optimization Update
The loss decreased from `1.0083` to a lower value. During backpropagation, the gradients update the target word embeddings `v_input` to align more closely with `v_target_ctx` and move away from the negative samples `v_neg_ctxs`, confirming successful parameter optimization.